# Orbitoby · 통합 물리 제원 + TLE · Colab

GCAT / ESA DISCOS / KeepTrack를 초급 API에서 **하나의 물리량 카탈로그**로 조회하고 NORAD ID로 TLE 이력과 선택적으로 결합합니다. 고도·이심률·관측기간 필터는 없습니다.

**필수 준비:** 이 기능이 포함된 Orbitoby 버전(신규 PyPI 배포판 또는 로컬 wheel), 그리고 Google Drive에 업로드한 `Orbitoby_Physical_Archives/physical_archives.sqlite3` 및 `catalog_all.csv`. 대형 TLE Parquet는 필요할 때만 Drive에 추가합니다. 제공기관 원본 데이터는 원저작권·라이선스 조건을 따라 관리하세요.


In [ ]:
# 새 기능이 PyPI에 공개된 뒤 사용하세요. 그 전에는 먼저 검증된 wheel을 설치해야 합니다.
%pip -q install --upgrade orbitoby duckdb pandas

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
from pathlib import Path

import orbitoby as ob

if not hasattr(ob, "physical_satellites"):
    raise RuntimeError(
        "현재 설치된 PyPI 버전에는 이 기능이 없습니다. PR 병합 및 새 PyPI 릴리스 이후 실행하거나 신규 wheel을 설치하세요."
    )

ROOT = Path("/content/drive/MyDrive/Orbitoby")
DB = ROOT / "Orbitoby_Physical_Archives"
CATALOG = ROOT / "catalog_all.csv"
assert (DB / "physical_archives.sqlite3").exists(), (
    "Mac 물리 제원 DB를 Drive로 복사하세요."
)
assert CATALOG.exists(), "전체 Orbitoby 카탈로그 CSV를 Drive에 저장하세요."
print("Ready:", DB)

In [ ]:
# 3개 출처 중 어느 하나가 각 물리량을 제공해도 충족한 것으로 분류
cohort = ob.physical_satellites(path=DB, catalog=CATALOG, required=["mass", "area"])
print("전체 모집단:", cohort.manifest["reference_catalog_norad_ids"])
print("질량+기하학적 단면적 위성:", len(cohort.table))
display(
    cohort.table[
        [
            "norad_id",
            "object_name",
            "mass_kg",
            "mass_definition",
            "geometric_area_m2",
            "has_source_conflict",
        ]
    ].head(20)
)

In [ ]:
# 출처별 세부 원본 확인 (값 충돌/시점 차이 감사)
with ob.physics(DB) as p:
    example = p.get(35694)
    print("원본 레코드:", len(example["records"]))
    display(
        p.satellites(
            catalog=CATALOG, required=["mass", "area"], include_sources=True
        ).table.head(5)
    )

In [ ]:
# 선택: 과거 TLE Parquet가 Drive에 준비된 경우만 실행 (대용량 읽기는 시간이 오래 걸릴 수 있음)
TLE_DIRECTORY = ROOT / "data"  # tle_YYYY.parquet가 있는 폴더
if TLE_DIRECTORY.exists() and list(TLE_DIRECTORY.glob("*.parquet")):
    joined = ob.physical_satellites(
        path=DB, catalog=CATALOG, required=["mass", "area"], tle_parquet=TLE_DIRECTORY
    )
    print(joined.manifest["tle_join"])
    display(
        joined.table[
            [
                "norad_id",
                "mass_kg",
                "geometric_area_m2",
                "tle_distinct_epochs",
                "has_tle_history",
            ]
        ].head()
    )
else:
    print("TLE 데이터 없음: 물리 제원 조회만 수행했습니다. (정상)")

In [ ]:
# CSV + 감사 manifest 저장
import json

out = ROOT / "Orbitoby_Combined_Physics_TLE"
out.mkdir(parents=True, exist_ok=True)
cohort.to_csv(out / "satellites_mass_plus_geometric_area.csv")
(out / "manifest.json").write_text(
    json.dumps(cohort.manifest, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)
print("Saved:", out)

**해석 주의:** `geometric_area_m2`는 ESA DISCOS의 기하학적 단면적이지 실제 대기항력에 대한 유효 투영면적은 아닙니다. `mass_kg`도 운용 시각의 정확한 질량이라고 보장되지 않습니다. 따라서 `mass/area` 후보가 나와도 완전한 탄도계수가 알려진 위성으로 간주하면 안 됩니다. TLE BSTAR와 RCS도 탄도계수로 자동 변환하지 않습니다.